## Set initial solution
Import libraries

In [ ]:
import os
import sys
import numpy as np
import tensorflow as tf

In [ ]:
sys.path.append(os.path.abspath('./../../tools/functions'))

from thermos import *
from writing import *

Define inputs

In [ ]:
# Freestream values
freestream = {
    'p': 1000.,
    'T': 500.,
    'u': 3000.,
    'X': {
        'O2': 0.95,
        'O': 0.05
    }
}
# Specie thermodynamics
species = ['O', 'O2']
specie_thermo = {
    0: {
        'specie': {
            'name': 'O',
            'model': {
               'levels': 'O3_UMN/O.csv'
            }
        },
        'thermo': {
            'model': 'Constant'
        }
    },
    1: {
        'specie': {
            'name': 'O2',
            'model': {
               'levels': 'O3_UMN/O2.csv',
               'lev_mapping': 'O3_UMN/O2/LevelsMap_RVE27.csv'
            }
        },
        'thermo': {
            'model': 'CoupledModes'
        }
    }
}

Initialize thermodynamics and extract primitive variables from PLATO solution

In [ ]:
freestream = tf.nest.map_structure(
    lambda x: np.array(x).reshape(1,1), freestream
)

In [ ]:
sp_th, mix = init_thermos(specie_thermo=specie_thermo)
sp_th, mix = update_thermos(sp_th, mix, freestream['X'], freestream['T'])

Write freestream condition to file

In [ ]:
comp_fs = []
for s in sp_th.values():
    if s.specie.n_comp == 1:
        comp = s.specie.name
    else:
        comp = [s.specie.name + f'_{i+1}' for i in range(s.specie.n_comp)]
    comp_fs.append(comp)
comp_fs = ' '.join(tf.nest.flatten(comp_fs))

file = './inputfile.sh'
if os.path.exists(file):
    replace(
        file=os.path.abspath(file),
        patterns=(
            "COMP_OO = ",
        ),
        replaces=(
            f"COMP_OO = {comp_fs}\n",
        )
    )

In [ ]:
X_fs = np.hstack([s.specie.X for s in sp_th.values()]).T
np.savetxt('Xi_oo.dat', X_fs)